In [ ]:
from selenium                     import webdriver
from selenium.webdriver.common.by import By

import os
import time
import pandas as pd
from tqdm import tqdm

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
# Configure Microsoft Edge browser options (start minimized)
options = webdriver.EdgeOptions()
options.add_argument('--start-minimized')

# Launch Edge browser with defined options and open Glassdoor job search page
driver = webdriver.Edge(options=options)
driver.get('https://www.glassdoor.co.in/Job/data-jobs-SRCH_KO0,4.htm')

In [ ]:
def close_login_popup(driver):
    """Close login popup if it appears."""
    try:
        # Locate and click the close button twice (sometimes required)
        close_login_popup = driver.find_element(By.CLASS_NAME, 'CloseButton')
        close_login_popup.click()
        close_login_popup.click()
    except:
        time.sleep(0)

In [ ]:
# Load multiple pages of job listings
for i in tqdm(range(0,25), desc='Progress'):
    close_login_popup(driver)
    try:
        time.sleep(3)
        all_matches_button = driver.find_element(By.CSS_SELECTOR, '[data-test="load-more"]') # Find "Show more jobs" button
    except:
        time.sleep(0)
        print('Show more jobs button not found')
    
    close_login_popup(driver)
    all_matches_button.click()

In [ ]:
job_dataset = pd.DataFrame()

# Collect all job listings on the page
close_login_popup(driver)
all_jobs_list = driver.find_elements(By.XPATH,"//li[@class='JobsList_jobListItem__JBBUV']")

# Process each job listing
for job in tqdm(all_jobs_list, desc='Progress'):
    job_record = {}
    
    close_login_popup(driver)
    job.click()
    time.sleep(2)
    
    try:
        show_more_button = driver.find_element(By.CLASS_NAME, 'JobDetails_showMore__j5Z_h')
        close_login_popup(driver)
        show_more_button.click()
    except:
        time.sleep(0)

    job_details_tab = driver.find_element(By.CLASS_NAME, 'JobDetails_jobDetailsContainer__sS1W1')

    try:
        company = job_details_tab.find_element(By.CLASS_NAME, 'EmployerProfile_employerName__Xemli')
        job_record['company'] = company.text
    except:
        job_record['company'] = ''
        
    try:
        job_title = job_details_tab.find_element(By.CLASS_NAME, 'JobDetails_jobTitle__Rw_gn')
        job_record['job_title'] = job_title.text
    except:
        job_record['job_title'] = ''

    try:
        company_rating = job_details_tab.find_element(By.CLASS_NAME, 'EmployerProfile_ratingContainer__N4hxE')
        job_record['company_rating'] = company_rating.text
    except:
        job_record['company_rating'] = ''
        
    try:
        job_description = job_details_tab.find_element(By.CLASS_NAME, 'JobDetails_showHidden__trRXQ')
        job_record['job_description'] = job_description.text
    except:
        job_record['job_description'] = ''

    try:
        location = job_details_tab.find_element(By.CLASS_NAME, 'JobDetails_location__MbnUM')
        job_record['location'] = location.text
    except:
        job_record['location'] = ''
        
    try:
        salary_avg_estimate = job_details_tab.find_element(By.CLASS_NAME, 'SalaryEstimate_averageEstimate__xF_7h')
        job_record['salary_avg_estimate'] = salary_avg_estimate.text
    except:
        job_record['salary_avg_estimate'] = ''

    try:
        salary_estimate_payperiod = job_details_tab.find_element(By.CLASS_NAME, 'SalaryEstimate_payPeriod__oBnsD')
        job_record['salary_estimate_payperiod'] = salary_estimate_payperiod.text
    except:
        job_record['salary_estimate_payperiod'] = ''

    try:
        company_overview_values = job_details_tab.find_elements(By.CLASS_NAME, 'JobDetails_overviewItemValue__5TqNi')
        job_record.update(
            {
                'company_size'   : company_overview_values[0].text,
                'company_founded': company_overview_values[1].text,
                'employment_type': company_overview_values[2].text,
                'industry'       : company_overview_values[3].text,
                'sector'         : company_overview_values[4].text,
                'revenue'        : company_overview_values[5].text
            }
        )
    except:
        job_record.update(
            {
                'company_size'   : '',
                'company_founded': '',
                'employment_type': '',
                'industry'       : '',
                'sector'         : '',
                'revenue'        : ''
            }
        )
    
    try:
        company_ratings = job_details_tab.find_elements(By.CLASS_NAME, 'JobDetails_ratingScore__Pg9_b')
        job_record['career_opportunities_rating'] = company_ratings[0].text
        job_record['comp_and_benefits_rating']    = company_ratings[1].text
        job_record['culture_and_values_rating']   = company_ratings[2].text
        job_record['senior_management_rating']    = company_ratings[3].text
        job_record['work_life_balance_rating']    = company_ratings[4].text
    except:
        job_record['career_opportunities_rating'] = ''
        job_record['comp_and_benefits_rating']    = ''
        job_record['culture_and_values_rating']   = ''
        job_record['senior_management_rating']    = ''
        job_record['work_life_balance_rating']    = ''
        time.sleep(0)
    
    job_dataset = job_dataset.append(job_record, ignore_index=True)

    print(job_record)
    print('\n')

In [ ]:
# Pause briefly before closing the browser
time.sleep(10)
driver.quit()

In [ ]:
job_dataset.head(5)

In [ ]:
job_dataset.to_csv(f"{DATA_DIR}/glassdoor_jobs.csv", index=False)